In [1]:
import numpy as np
import pandas as pd
import os

from preprocessing.mimic_preprocessing.database_assembly.further_exclusion_criteria import apply_further_exclusion_criteria

In [2]:
DATA_DIR = '/mnt/data1/klug/datasets/opsum/short_term_outcomes/with_imaging/mimic_prepro_16022026_095909'

admission_notes_path = os.path.join(DATA_DIR, 'original_tables/combined_notes_labels_v2.xlsx')
admission_table_path = os.path.join(DATA_DIR, 'original_tables/admission_df.csv')
monitoring_path = os.path.join(DATA_DIR, 'original_tables/monitoring_df.csv')
outcomes_short_term_path = os.path.join(DATA_DIR, 'preprocessed_outcomes_short_term_16022026_095909.csv')
outcomes_path = os.path.join(DATA_DIR, 'preprocessed_outcomes_16022026_095909.csv')
features_path = os.path.join(DATA_DIR, 'preprocessed_features_16022026_095909.csv')

In [3]:
admission_data_df = pd.read_excel(admission_notes_path)
admission_table_df = pd.read_csv(admission_table_path)
monitoring_df = pd.read_csv(monitoring_path)
outcomes_short_term_df = pd.read_csv(outcomes_short_term_path)
outcomes_df = pd.read_csv(outcomes_path)
features_df = pd.read_csv(features_path)

/tmp/ipykernel_680554/2501204034.py:3: DtypeWarning: Columns (8,10) have mixed types. Specify dtype option on import or set low_memory=False.
  monitoring_df = pd.read_csv(monitoring_path)


In [4]:
# Final cohort: patients that passed all preprocessing exclusion criteria
cohort_cids = features_df['case_admission_id'].unique()
print(f'Number of unique patients in features cohort: {len(cohort_cids)}')

Number of unique patients in features cohort: 247


In [5]:
# Filter admission notes: admitted to ICU for stroke, onset to admission <= 7d
admission_data_df = admission_data_df[admission_data_df['admitted to ICU for stroke'] == 'y']
admission_data_df = admission_data_df[admission_data_df['onset to ICU admission > 7d'] == 'n']
admission_data_df['case_admission_id'] = (
    admission_data_df['hadm_id'].astype(str) + '_' + admission_data_df['icustay_id'].astype(str)
)

# Apply further exclusion criteria
hadm_ids_to_exclude = apply_further_exclusion_criteria(
    admission_data_df['case_admission_id'].unique(),
    admission_table_path,
    admission_notes_path,
    log_dir=''
)
admission_data_df = admission_data_df[~admission_data_df['case_admission_id'].isin(hadm_ids_to_exclude)]

# Extract BMI from monitoring data
height_labels = ['Height (cm)', 'Height', 'Admit Ht']
weight_labels = ['Admit Wt', 'Admission Weight (lbs.)', 'Admission Weight (Kg)',
                 'Previous WeightF', 'Previous Weight', 'Daily Weight']
monitoring_df['case_admission_id'] = (
    monitoring_df['hadm_id'].astype(int).astype(str) + '_' + monitoring_df['icustay_id'].astype(int).astype(str)
)
monitoring_df = monitoring_df[monitoring_df['case_admission_id'].isin(admission_data_df['case_admission_id'].unique())]

height_df = monitoring_df[monitoring_df['label'].isin(height_labels)].dropna(subset=['valuenum']).copy()
height_df.loc[height_df['valueuom'].isin(['inches', 'Inch']), 'valuenum'] = (
    height_df.loc[height_df['valueuom'].isin(['inches', 'Inch']), 'valuenum'] * 2.54
)
height_df = (height_df.sort_values(by=['charttime'])
             .groupby('case_admission_id').first().reset_index()[['case_admission_id', 'valuenum']])
height_df.rename(columns={'valuenum': 'height'}, inplace=True)

weight_df = monitoring_df[monitoring_df['label'].isin(weight_labels)].dropna(subset=['valuenum'])
weight_df = (weight_df.sort_values(by=['charttime'])
             .groupby('case_admission_id').first().reset_index()[['case_admission_id', 'valuenum']])
weight_df.rename(columns={'valuenum': 'weight'}, inplace=True)

bmi_df = pd.merge(height_df, weight_df, on='case_admission_id', how='inner')
bmi_df['BMI'] = bmi_df['weight'] / (bmi_df['height'] / 100) ** 2
admission_data_df = pd.merge(admission_data_df, bmi_df[['case_admission_id', 'BMI']], on='case_admission_id', how='left')

# Extract data from admission table (age, sex)
admission_table_df = admission_table_df[
    ['subject_id', 'hadm_id', 'icustay_id', 'dob', 'admittime', 'age', 'gender', 'admission_location']
]
admission_table_df['case_admission_id'] = (
    admission_table_df['hadm_id'].astype(str) + '_' + admission_table_df['icustay_id'].astype(str)
)
admission_table_df.drop_duplicates(inplace=True)
admission_table_df.loc[admission_table_df['age'] > 250, 'age'] = 90
admission_table_df.rename(columns={'gender': 'Sex'}, inplace=True)
admission_table_df.loc[admission_table_df.Sex == 'F', 'Sex'] = 'Female'
admission_table_df.loc[admission_table_df.Sex == 'M', 'Sex'] = 'Male'

admission_data_df = pd.merge(
    admission_data_df,
    admission_table_df[['case_admission_id', 'age', 'Sex']],
    on='case_admission_id', how='left'
)

# Derive IVT/IAT from time columns
admission_data_df['IVT with rtPA'] = ~admission_data_df['IVT time'].isna()
admission_data_df['IAT'] = ~admission_data_df['IAT time'].isna()

# Join long-term outcomes
outcomes_df_sub = outcomes_df[['case_admission_id', 'Death in hospital', '3M Death']].drop_duplicates()
admission_data_df = pd.merge(admission_data_df, outcomes_df_sub, on='case_admission_id', how='left')

# Align column names
admission_data_df.rename(columns={
    'age': 'Age (calc.)',
    'prestroke mRS': 'Prestroke disability (Rankin)',
    'admission NIHSS': 'NIH on admission'
}, inplace=True)

# Convert categorical encodings
medhist_columns = ['MedHist Hypertension', 'MedHist Diabetes', 'MedHist Hyperlipidemia', 'MedHist Atrial Fibr.']
admission_data_df[medhist_columns] = admission_data_df[medhist_columns].replace({'y': 'yes', 'n': 'no'})
if 'MedHist Smoking' in admission_data_df.columns:
    admission_data_df['MedHist Smoking'] = admission_data_df['MedHist Smoking'].replace({'y': 'yes', 'n': 'no'})
admission_data_df[['IVT with rtPA', 'IAT']] = admission_data_df[['IVT with rtPA', 'IAT']].replace(
    {True: 'yes', False: 'no'}
)
admission_data_df[['Death in hospital', '3M Death']] = admission_data_df[['Death in hospital', '3M Death']].replace(
    {1: 'yes', 0: 'no'}
)

# Restrict to patients in the features cohort
patient_df = admission_data_df[admission_data_df['case_admission_id'].isin(cohort_cids)].copy()
print(f'Number of patients in final cohort: {len(patient_df)}')

Number of patients in final cohort: 247


In [6]:
end_cids = set(outcomes_short_term_df['case_admission_id'].unique())
patient_df['END'] = 0
patient_df.loc[patient_df['case_admission_id'].isin(end_cids), 'END'] = 1
print(f'Patients with END: {patient_df["END"].sum()}, without END: {(patient_df["END"] == 0).sum()}')

Patients with END: 139, without END: 108


In [7]:
CONTINUOUS_CHARACTERISTICS = [
    'Age (calc.)',
    'Prestroke disability (Rankin)',
    'NIH on admission',
    'BMI'
]

CATEGORICAL_CHARACTERISTICS = [
    'Sex',
    'IVT with rtPA',
    'IAT',
    'MedHist Hypertension',
    'MedHist Diabetes',
    'MedHist Hyperlipidemia',
    'MedHist Atrial Fibr.',
    'MedHist Smoking',
    '3M Death'
]

# Remove characteristics not present in the data
CATEGORICAL_CHARACTERISTICS = [c for c in CATEGORICAL_CHARACTERISTICS if c in patient_df.columns]

In [8]:
def create_population_table(df, continuous_characteristics, categorical_characteristics, count_nan=False):
    """
    Create a population descriptive table from a dataframe.
    Continuous variables: median (Q1-Q3)
    Categorical variables: n (X.X%)
    """
    population_df = pd.DataFrame()
    population_str_df = pd.DataFrame()

    n_cases = df.case_admission_id.nunique()
    n_patients = df.case_admission_id.apply(lambda x: x.split('_')[0]).nunique()

    population_df['n cases'] = [n_cases]
    population_df['n patients'] = [n_patients]
    population_str_df['n patients'] = [n_patients]
    population_str_df['n cases'] = [n_cases]

    for characteristic in continuous_characteristics:
        population_df[f'median {characteristic}'] = [df[characteristic].median()]
        population_df[f'Q25 {characteristic}'] = [df[characteristic].quantile(0.25)]
        population_df[f'Q75 {characteristic}'] = [df[characteristic].quantile(0.75)]
        population_df[f'n missing {characteristic}'] = [df[characteristic].isnull().sum()]
        population_str_df[f'{characteristic}'] = (
            f'{population_df[f"median {characteristic}"][0]:.1f} '
            f'({population_df[f"Q25 {characteristic}"][0]:.1f}-'
            f'{population_df[f"Q75 {characteristic}"][0]:.1f})'
        )

    for characteristic in categorical_characteristics:
        max_category = df[characteristic].value_counts().idxmax()
        if 'yes' in df[characteristic].unique():
            display_category = 'yes'
        elif 1 in df[characteristic].unique():
            display_category = 1
        elif 'Female' in df[characteristic].unique():
            display_category = 'Female'
        else:
            display_category = max_category
        population_df[f'{characteristic} {display_category}'] = [
            df[characteristic].value_counts()[display_category]
        ]
        if not count_nan:
            population_df[f'p {characteristic} {display_category}'] = [
                df[characteristic].value_counts()[display_category] / df[characteristic].count() * 100
            ]
        else:
            population_df[f'p {characteristic} {display_category}'] = [
                df[characteristic].value_counts()[display_category] / len(df) * 100
            ]
        population_df[f'n missing {characteristic}'] = [df[characteristic].isnull().sum()]

        if display_category == 'yes' or display_category == 1:
            population_str_df[f'{characteristic}'] = (
                f'{population_df[f"{characteristic} {display_category}"][0]} '
                f'({population_df[f"p {characteristic} {display_category}"][0]:.1f}%)'
            )
        else:
            population_str_df[f'{characteristic} ({display_category})'] = (
                f'{population_df[f"{characteristic} {display_category}"][0]} '
                f'({population_df[f"p {characteristic} {display_category}"][0]:.1f}%)'
            )

    return population_df, population_str_df

In [9]:
overall_population_df, overall_population_str_df = create_population_table(
    patient_df, CONTINUOUS_CHARACTERISTICS, CATEGORICAL_CHARACTERISTICS
)
end_population_df, end_population_str_df = create_population_table(
    patient_df[patient_df.END == 1], CONTINUOUS_CHARACTERISTICS, CATEGORICAL_CHARACTERISTICS
)
no_end_population_df, no_end_population_str_df = create_population_table(
    patient_df[patient_df.END == 0], CONTINUOUS_CHARACTERISTICS, CATEGORICAL_CHARACTERISTICS
)

comparison_table_df = pd.concat([overall_population_str_df.T, end_population_str_df.T, no_end_population_str_df.T], axis=1)
comparison_table_df.columns = ['Overall', 'END', 'No END']

In [10]:
comparison_table_df

,Overall,END,No END
n patients,247,139,108
n cases,247,139,108
Age (calc.),73.1 (58.5-82.1),76.6 (63.7-84.6),66.2 (53.1-78.4)
Prestroke disability (Rankin),2.0 (1.0-2.5),2.0 (1.0-3.0),1.0 (0.0-2.0)
NIH on admission,13.0 (7.0-19.0),15.0 (9.0-19.0),11.0 (5.0-17.0)
BMI,30.7 (25.6-48.5),30.7 (24.9-52.4),30.5 (25.9-37.2)
Sex (Female),125 (50.6%),72 (51.8%),53 (49.1%)
IVT with rtPA,159 (64.4%),83 (59.7%),76 (70.4%)
IAT,50 (20.2%),34 (24.5%),16 (14.8%)
MedHist Hypertension,168 (68.0%),98 (70.5%),70 (64.8%)


In [11]:
comparison_table_df.to_csv(os.path.join(DATA_DIR, 'mimic_end_table1.csv'))